In [1]:
import pandas as pd
import numpy as np
import re
from typing import Tuple, List
from scipy.stats import spearmanr

import warnings
warnings.filterwarnings("ignore")

In [2]:
def load_and_process_data(in_vitro: str = "data/in_vitro_modeling.csv", marmoset: str = "data/marmoset_wide_clustered_classif.csv") -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load and process data for in vitro and marmoset datasets."""
    try: 
        in_vitro = pd.read_csv(in_vitro)
        marmoset = pd.read_csv(marmoset)
        
        print(f"In vitro data shape: {in_vitro.shape}")
        print(f"Marmoset data shape: {marmoset.shape}")
        
        return in_vitro, marmoset

    except Exception as e:
        print(f"Error loading data: {e}")
        return None, None

In [3]:
def identify_pathology_features(marmoset_df: pd.DataFrame) -> Tuple[List[str], List[str], List[str]]:
    """
    Identify pathology features in marmoset dataset.
    Returns a list of:
    - All common features between TP2 and TP6
    - TP2 feature columns
    - TP6 feature columns
    """
    
    all_cols = marmoset_df.columns.tolist()
    
    # feature selection
    tp2_cols = [c for c in all_cols if c.startswith('TP2_') and not any(x in c for x in ['LesionType', 'FinalLesionType', 'Consolidate', 'Contour'])]
    tp6_cols = [c for c in all_cols if c.startswith('TP6_') and not any(x in c for x in ['LesionType', 'FinalLesionType', 'Consolidate', 'Contour'])]

    # base feature names
    tp2_base = [c.replace('TP2_', '') for c in tp2_cols]
    tp6_base = [c.replace('TP6_', '') for c in tp6_cols]

    # common features
    common_features = list(set(tp2_base) & set(tp6_base))
    
    print(f"Found {len(common_features)} common features between TP2 and TP6.")
    
    return common_features, tp2_cols, tp6_cols

In [4]:
def calculate_deltas(marmoset_df: pd.DataFrame, common_features: List[str]) -> pd.DataFrame:
    """
    Calculate deltas (TP6-TP2) for pathology features.
    Deltas are calculated in such a way that negative values indicate improvement.
    """
    
    delta_df = marmoset_df.copy()
    
    for feature in common_features:
        tp6_col = f'TP6_{feature}'
        tp2_col = f'TP2_{feature}'
        delta_col = f'delta_{feature}'
        
        if tp6_col in marmoset_df.columns and tp2_col in marmoset_df.columns:
            delta_df[delta_col] = marmoset_df[tp6_col] - marmoset_df[tp2_col]
    
    return delta_df

In [5]:
def get_correlation_features(in_vitro_df: pd.DataFrame, marmoset_df: pd.DataFrame, common_features: List[str]) -> Tuple[List[str], List[str]]:
    """Get lists of features for correlation analysis."""
    
    # in vitro features
    in_vitro_features = [col for col in in_vitro_df.columns if col != 'Drug']
    
    # TP6 features
    tp6_features = [f'TP6_{feature}' for feature in common_features]
    
    # TP2 features
    tp2_features = [f'TP2_{feature}' for feature in common_features]
    
    # delta features
    delta_features = [f'delta_{feature}' for feature in common_features]
    
    # all pathology features
    all_pathology_features = tp6_features + delta_features # TP2 is start of treatment so meaningless for correlations
    
    print(f"In vitro features: {len(in_vitro_features)}")
    print(f"TP6 features: {len(tp6_features)}")
    print(f"Delta features: {len(delta_features)}")
    
    return in_vitro_features, all_pathology_features

In [6]:
def filter_in_vitro_features(features: List[str],
                            exclude_prefixes: List[str] = None,
                            exclude_substrings: List[str] = None) -> List[str]:
    exclude_prefixes = exclude_prefixes or []
    exclude_substrings = exclude_substrings or []
    out = []
    
    for f in features:
        if any(f.startswith(p) for p in exclude_prefixes):
            continue
        if any (s in f for s in exclude_substrings):
            continue
        out.append(f)
    
    return out

In [7]:
def calculate_correlations_by_severity(in_vitro_df: pd.DataFrame, delta_df: pd.DataFrame, in_vitro_features: List[str], pathology_features: List[str], severity: str) -> Tuple[np.ndarray, np.ndarray]:
    """Calculates Spearman correlations for a specific severity class."""
    
    # filter by severity
    severity_data = delta_df[delta_df['classif'] == severity].copy()
    
    if len(severity_data) == 0:
        raise ValueError(f"No data found for severity: {severity}. Options are: {delta_df['classif'].unique()}.")
    
    print(f"\nCalculating correlations for {severity} lesions (n={len(severity_data)})")
    
    # initialize results matrices
    rho_matrix = np.full((len(in_vitro_features), len(pathology_features)), np.nan)
    p_matrix = np.full((len(in_vitro_features), len(pathology_features)), np.nan)
    
    # compounds from both datasets
    in_vitro_compounds = set(in_vitro_df['Drug'].str.upper())
    marmoset_compounds = set(severity_data['Compound'].str.upper())
    common_compounds = in_vitro_compounds & marmoset_compounds
    
    print(f"Common compounds for correlation analysis: {sorted(common_compounds)}")
    
    if len(common_compounds) < 3:
        print(f"[WARNING]: Only {len(common_compounds)} common compounds found for {severity} severity. Results may be unstable.")
        return None, None
    
    for i, iv_feature in enumerate(in_vitro_features):
        for j, path_feature in enumerate(pathology_features):
            try:
                merged_data = []
                
                for compound in common_compounds:
                    iv_row = in_vitro_df[in_vitro_df['Drug'].str.upper() == compound]
                    if len(iv_row) == 0 or pd.isna(iv_row[iv_feature].iloc[0]):
                        continue
                    iv_value = iv_row[iv_feature].iloc[0]
                    
                    path_rows = severity_data[severity_data['Compound'].str.upper() == compound]
                    for _, path_row in path_rows.iterrows():
                        if pd.notna(path_row[path_feature]):
                            merged_data.append({
                                'iv_value': iv_value,
                                'path_value': path_row[path_feature]
                            })
                            
                    if len(merged_data) > 3:
                        iv_values = [d['iv_value'] for d in merged_data]
                        path_values = [d['path_value'] for d in merged_data]
                        
                        rho, p_value = spearmanr(iv_values, path_values)
                        rho_matrix[i, j] = rho
                        p_matrix[i, j] = p_value
                        
            except Exception as e:
                print(f"Error calculating correlation for {iv_feature} vs {path_feature}: {e}")
                continue
            
    return rho_matrix, p_matrix

In [8]:
def save_correlation_tables(rho_matrix: np.ndarray, p_matrix: np.ndarray, in_vitro_features: List[str], pathology_features: List[str], severity_name: str) -> None:
    """Save correlation results to a .csv."""

    if rho_matrix is None or p_matrix is None:
        print(f"No valid correlations calculated for {severity} lesions.")
        return
    
    rho_df = pd.DataFrame(rho_matrix,
                        index=in_vitro_features,
                        columns=pathology_features)
    rho_df.index.name = 'in_vitro_feature'
    
    p_df = pd.DataFrame(p_matrix,
                        index=in_vitro_features,
                        columns=pathology_features)
    p_df.index.name = 'in_vitro_feature'
    
    # save files
    rho_filename = f"{severity_name}_rho_values.csv"
    p_filename = f"{severity_name}_p_values.csv"
    
    rho_df.to_csv(rho_filename)
    p_df.to_csv(p_filename)
    
    print(f"Saved {rho_filename} and {p_filename}")
    
    # summary statistics
    valid_correlations = np.sum(~np.isnan(rho_matrix))
    significant_correlations = np.sum((p_matrix < 0.05) & (~np.isnan(p_matrix)))
    
    print(f"Summary for {severity} lesions: ")
    print(f"     Valid corelations: {valid_correlations}")
    print(f"     Significantly correlated: {significant_correlations}")
    
    if valid_correlations > 0:
        print(f"     Correlation range: {np.nanmin(rho_matrix):3f} to {np.nanmax(rho_matrix):3f}")

## Correlation Directionality:
- IN VITRO FEATURES:
    - FxC: negative = synergistic (good) | positive = antagonistic (bad)
    - AUC: closer to 0 = more potent (good) | closer to 1 = less potent (bad)
    - GRinf: negative = good | positive = bad
    - Einf: closer to 1 = more potent (good) | closer to 0 = less potent (bad)

- PATHOLOGY FEATURES:
    - TP6 values: lower = better outcome (good) | higher = worse outcome (bad)
    - Delta (TP6-TP2): negative = improvement (good) | positive = worsening (bad)
    - CFU: closer to 0 = less bugs (good) | higher = more bugs (bad)

For a MATCHING correlation (good in vitro correlated with good outcome):
    - Case 1: FxC, AUC, GRinf: a positive correlation is good (both decrease together)
    - Case 2: Einf: a negative correlation is good (Einf increases, TP6 decreases)

In [9]:
IN_VITRO_DF = "data/in_vitro_combos.csv"
MARMOSET_DF = "data/marm_data_wide_clustered_classif.csv"

in_vitro_df, marmoset_df = load_and_process_data(IN_VITRO_DF, MARMOSET_DF)

common_features, tp2_cols, tp6_cols = identify_pathology_features(marmoset_df)
delta_df = calculate_deltas(marmoset_df, common_features)

in_vitro_features, pathology_features = get_correlation_features(in_vitro_df, delta_df, common_features)

in_vitro_features = filter_in_vitro_features(in_vitro_features,
                                            exclude_prefixes=["IC50", "IC90", 
                                                            "FBC50" ,"FBC90",
                                                            "LoeweFIC50", "LoeweFIC90",
                                                            ])

for severity in ['cool', 'hot']:
    severity_canon = 'less_severe' if severity == 'cool' else 'severe'
    print(f"Analyzing {severity_canon} lesions...")
    
    rho_matrix, p_matrix = calculate_correlations_by_severity(
        in_vitro_df, delta_df, in_vitro_features, pathology_features, severity
    )
    
    save_correlation_tables(rho_matrix, p_matrix, in_vitro_features, pathology_features, severity_canon)

In vitro data shape: (10, 271)
Marmoset data shape: (1193, 129)
Found 17 common features between TP2 and TP6.
In vitro features: 270
TP6 features: 17
Delta features: 17
Analyzing less_severe lesions...

Calculating correlations for cool lesions (n=627)
Common compounds for correlation analysis: ['BDQ+DEL', 'BDQ+LIN', 'BDQ+LIN+PRE', 'BDQ+PRE', 'EMB+INH+PZA+RIF', 'EMB+MOX+PZA+RIF', 'INH+PZA', 'LIN+PRE', 'MOX+RIF', 'PZA+RIF']
Saved less_severe_rho_values.csv and less_severe_p_values.csv
Summary for cool lesions: 
     Valid corelations: 4752
     Significantly correlated: 1621
     Correlation range: -0.463592 to 0.603591
Analyzing severe lesions...

Calculating correlations for hot lesions (n=566)
Common compounds for correlation analysis: ['BDQ+DEL', 'BDQ+LIN', 'BDQ+LIN+PRE', 'BDQ+PRE', 'EMB+INH+PZA+RIF', 'EMB+MOX+PZA+RIF', 'INH+PZA', 'LIN+PRE', 'MOX+RIF', 'PZA+RIF']
Saved severe_rho_values.csv and severe_p_values.csv
Summary for hot lesions: 
     Valid corelations: 4752
     Significa

In [10]:
test_df = pd.read_csv("data/in_vitro_modeling.csv")

test_df['Drug'].unique()

array(['BDQ', 'BDQ+DEL', 'BDQ+LIN', 'BDQ+LIN+PRE', 'BDQ+PRE', 'DEL',
       'EMB+INH+PZA+RIF', 'EMB+MOX+PZA+RIF', 'INH', 'INH+PZA', 'LIN',
       'LIN+PRE', 'MOX', 'MOX+RIF', 'PRE', 'PZA', 'PZA+RIF', 'RIF'],
      dtype=object)